# 14 · Familia BERT: BETO

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Ajuste fino con LoRA (r = 8, α = 16 en query y value), pérdida ponderada por clase y 3 épocas; evaluación sobre la prueba final. Esta ejecución corresponde a BETO.

*Nombre durante el desarrollo: `09_BERT_Family_Split_Oficial` (algunas salidas conservan esa numeración).*

## 1. Instalar dependencias y descargar el split oficial

In [1]:
!pip install -q -U transformers accelerate peft "huggingface_hub<2.0" scikit-learn "pandas<3" pyarrow
!pip uninstall -y -q torchao

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"
REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"
REPO_CHECKPOINTS = f"{TU_USUARIO_HF}/emotion-futbol-checkpoints"

import pandas as pd
import numpy as np
import torch

ruta_train = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename="04b_entrenamiento_72k.csv")
ruta_val_interna = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename="04b_validacion_interna_18k.csv")
ruta_test = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename="final/prueba_final_10k.csv")

df_train = pd.read_csv(ruta_train)
df_val_interna = pd.read_csv(ruta_val_interna)
df_test = pd.read_csv(ruta_test)

print(f"Entrenamiento: {len(df_train)} | Validación interna: {len(df_val_interna)} | Test final: {len(df_test)}")

SEED = 42
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
label2id = {e: i for i, e in enumerate(EMOCIONES)}
id2label = {i: e for i, e in enumerate(EMOCIONES)}
COLUMNA_TEXTO = "text"

CANDIDATOS = {
    "robertuito": "pysentimiento/robertuito-base-uncased",
    "beto": "dccuchile/bert-base-spanish-wwm-cased",
}
MODELO_A_CORRER = "beto"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


04b_entrenamiento_72k.csv:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

04b_validacion_interna_18k.csv:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

prueba_final_10k.csv:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Entrenamiento: 72000 | Validación interna: 18000 | Test final: 10000


## 2. Dataset, colector de resultados y utilidades

In [2]:
import os, time, io
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score

RUTA_RESULTADOS = "09_resultados_bert_family.csv"

class ComentariosDatasetHF(torch.utils.data.Dataset):
    def __init__(self, textos, etiquetas, tokenizer, max_len=64):
        self.encodings = tokenizer(list(textos), truncation=True, max_length=max_len)
        self.labels = [label2id[e] for e in etiquetas]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {"accuracy": acc, "precision_macro": precision, "recall_macro": recall, "f1_macro": f1}

def tamano_adaptador_mb(ruta):
    return sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fnames in os.walk(ruta) for f in fnames
    ) / (1024 ** 2)

def registrar_resultado(nombre, y_true, y_pred, latencia_prom_seg=None, memoria_pico_mb=None,
                         almacenamiento_modelo_mb=None):
    reporte = classification_report(y_true, y_pred, labels=EMOCIONES, output_dict=True, zero_division=0)
    fila = {
        "modelo": nombre,
        "n_test": len(y_true),
        "accuracy": reporte["accuracy"],
        "precision_macro": reporte["macro avg"]["precision"],
        "recall_macro": reporte["macro avg"]["recall"],
        "f1_macro": reporte["macro avg"]["f1-score"],
        "precision_weighted": reporte["weighted avg"]["precision"],
        "recall_weighted": reporte["weighted avg"]["recall"],
        "f1_weighted": reporte["weighted avg"]["f1-score"],
        "latencia_prom_seg": latencia_prom_seg,
        "memoria_pico_mb": memoria_pico_mb,
        "almacenamiento_modelo_mb": almacenamiento_modelo_mb,
    }
    for e in EMOCIONES:
        fila[f"f1_{e}"] = reporte.get(e, {}).get("f1-score", None)

    df_fila = pd.DataFrame([fila])
    if os.path.exists(RUTA_RESULTADOS):
        df_res = pd.read_csv(RUTA_RESULTADOS)
        df_res = df_res[df_res['modelo'] != nombre]
        df_res = pd.concat([df_res, df_fila], ignore_index=True)
    else:
        df_res = df_fila
    df_res.to_csv(RUTA_RESULTADOS, index=False)

    print(classification_report(y_true, y_pred, labels=EMOCIONES, zero_division=0))
    print(f"Guardado en {RUTA_RESULTADOS}. Modelos registrados: {sorted(df_res['modelo'].tolist())}")

## 3. Cargar modelo base + aplicar LoRA

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType

modelo_id = CANDIDATOS[MODELO_A_CORRER]
print(f"Entrenando: {MODELO_A_CORRER} -> {modelo_id}")

tokenizer = AutoTokenizer.from_pretrained(modelo_id)

ds_train = ComentariosDatasetHF(df_train[COLUMNA_TEXTO], df_train['label'], tokenizer)
ds_val_interna = ComentariosDatasetHF(df_val_interna[COLUMNA_TEXTO], df_val_interna['label'], tokenizer)
ds_test = ComentariosDatasetHF(df_test[COLUMNA_TEXTO], df_test['label'], tokenizer)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

base_model = AutoModelForSequenceClassification.from_pretrained(
    modelo_id, num_labels=len(EMOCIONES), id2label=id2label, label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "value"],
)
modelo = get_peft_model(base_model, lora_config)
modelo.print_trainable_parameters()

Entrenando: beto -> dccuchile/bert-base-spanish-wwm-cased


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

trainable params: 300,295 || all params: 110,156,558 || trainable%: 0.2726


## 4. Pesos de clase

In [4]:
from collections import Counter

conteo_clases = Counter(df_train['label'])
pesos_clase = torch.tensor(
    [len(df_train) / (len(EMOCIONES) * conteo_clases[e]) for e in EMOCIONES],
    dtype=torch.float32,
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, pesos_clase=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pesos_clase = pesos_clase

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        criterio = torch.nn.CrossEntropyLoss(weight=self.pesos_clase.to(logits.device))
        loss = criterio(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("Pesos de clase:", dict(zip(EMOCIONES, pesos_clase.tolist())))

Pesos de clase: {'alegria': 0.4340513348579407, 'tristeza': 2.9909026622772217, 'enojo': 0.4614497125148773, 'miedo': 54.711246490478516, 'sorpresa': 3.492602586746216, 'burla': 0.878145158290863, 'neutral': 1.3309671878814697}


## 5. Entrenar sobre las 72k, monitorear con las 18k de validación interna

In [5]:
carpeta_checkpoints = f"./checkpoints_{MODELO_A_CORRER}"

args = TrainingArguments(
    output_dir=carpeta_checkpoints,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    logging_steps=100,
    seed=SEED,
)

trainer = WeightedTrainer(
    model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val_interna,
    data_collator=collator, compute_metrics=compute_metrics, pesos_clase=pesos_clase,
)

tiene_checkpoint = os.path.isdir(carpeta_checkpoints) and any(
    d.startswith("checkpoint") for d in os.listdir(carpeta_checkpoints)
)
if tiene_checkpoint:
    print(f"Retomando desde checkpoint guardado en {carpeta_checkpoints}")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

trainer.train(resume_from_checkpoint=tiene_checkpoint)

resultado_eval_interna = trainer.evaluate()
print(f"\n=== {MODELO_A_CORRER}: métricas sobre la validación interna (18k, monitoreo) ===")
print(resultado_eval_interna)

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.129426,1.110046,0.655333,0.563923,0.585090,0.560889
2,1.054320,1.054413,0.656111,0.551450,0.602145,0.560447
3,0.913472,1.039231,0.667167,0.551896,0.617695,0.570925


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.913472,1.039231,3,0.667167,0.551896,0.617695,0.570925



=== beto: métricas sobre la validación interna (18k, monitoreo) ===
{'eval_loss': 1.0392305850982666, 'eval_accuracy': 0.6671666666666667, 'eval_precision_macro': 0.5518962781419207, 'eval_recall_macro': 0.6176950140713927, 'eval_f1_macro': 0.5709247252644603}


## 6. Evaluación sobre la prueba final

In [6]:
inicio_inf = time.time()
predicciones = trainer.predict(ds_test)
latencia_prom_seg = (time.time() - inicio_inf) / len(df_test)
memoria_pico_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

preds_ids = np.argmax(predicciones.predictions, axis=1)
pred_labels = [id2label[i] for i in preds_ids]

ruta_adaptador = f"./lora_{MODELO_A_CORRER}"
modelo.save_pretrained(ruta_adaptador)
almacenamiento_mb = tamano_adaptador_mb(ruta_adaptador)

print(f"=== {MODELO_A_CORRER}: evaluación final sobre el test oficial (10k) ===")
registrar_resultado(
    MODELO_A_CORRER, df_test['label'].tolist(), pred_labels,
    latencia_prom_seg=latencia_prom_seg, memoria_pico_mb=memoria_pico_mb,
    almacenamiento_modelo_mb=almacenamiento_mb,
)

del modelo, base_model, trainer
torch.cuda.empty_cache()

=== beto: evaluación final sobre el test oficial (10k) ===
              precision    recall  f1-score   support

     alegria       0.82      0.70      0.75      3282
    tristeza       0.44      0.71      0.54       479
       enojo       0.80      0.67      0.73      3104
       miedo       0.48      0.38      0.43        26
    sorpresa       0.35      0.61      0.44       409
       burla       0.69      0.68      0.69      1633
     neutral       0.49      0.67      0.56      1067

    accuracy                           0.68     10000
   macro avg       0.58      0.63      0.59     10000
weighted avg       0.72      0.68      0.69     10000

Guardado en 09_resultados_bert_family.csv. Modelos registrados: ['beto']


## 7. Subir el adaptador LoRA entrenado a Hugging Face

In [7]:
from huggingface_hub import upload_folder

ruta_en_hub = f"final/bert_family/{MODELO_A_CORRER}"
upload_folder(
    folder_path=ruta_adaptador,
    path_in_repo=ruta_en_hub,
    repo_id=REPO_CHECKPOINTS,
    repo_type="dataset",
)
print(f"Adaptador de '{MODELO_A_CORRER}' subido a Hugging Face en: {REPO_CHECKPOINTS}/{ruta_en_hub}")

Adaptador de 'beto' subido a Hugging Face en: santiagortegasegura/emotion-futbol-checkpoints/final/bert_family/beto


## 8. Subir `09_resultados_bert_family.csv` a Hugging Face

In [8]:
from huggingface_hub import upload_file, hf_hub_download as _hf_hub_download
try:
    from huggingface_hub.errors import EntryNotFoundError
except ImportError:
    from huggingface_hub.utils import EntryNotFoundError

df_res_local = pd.read_csv(RUTA_RESULTADOS)

try:
    ruta_remota_previa = _hf_hub_download(
        repo_id=REPO_DATOS, repo_type="dataset", filename="final/09_resultados_bert_family.csv"
    )
    df_remoto_previo = pd.read_csv(ruta_remota_previa)
    df_res_local = pd.concat([df_remoto_previo, df_res_local], ignore_index=True)
    df_res_local = df_res_local.drop_duplicates(subset="modelo", keep="last")
    df_res_local.to_csv(RUTA_RESULTADOS, index=False)
    print(f"Combinado con la versión previa en Hugging Face. Modelos en total: {sorted(df_res_local['modelo'].tolist())}")
except EntryNotFoundError:
    print("Todavía no existe una versión previa en Hugging Face -- se sube esta como la primera.")

upload_file(
    path_or_fileobj=RUTA_RESULTADOS,
    path_in_repo="final/09_resultados_bert_family.csv",
    repo_id=REPO_DATOS,
    repo_type="dataset",
)
print("09_resultados_bert_family.csv subido a tu bodega de datos en Hugging Face.")

09_resultados_bert_family.csv:   0%|          | 0.00/554 [00:00<?, ?B/s]

Combinado con la versión previa en Hugging Face. Modelos en total: ['beto', 'robertuito']
09_resultados_bert_family.csv subido a tu bodega de datos en Hugging Face.
